<a href="https://colab.research.google.com/github/Rudhra-06/Rudhrashini-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Day_04_Machine_Learning_and_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

a=pd.read_csv("/content/SOCR-HeightWeight.csv")

a.head()

,Index,Height(Inches),Weight(Pounds)
0,1,65.78331,112.9925
1,2,71.51521,136.4873
2,3,69.39874,153.0269
3,4,68.21660,142.3354
4,5,67.78781,144.2971


In [2]:
x=a[['Height(Inches)']]
y=a[['Weight(Pounds)']]

In [3]:
a.isnull().any()

,0
Index,False
Height(Inches),False
Weight(Pounds),False


In [4]:
a.isnull().sum()

,0
Index,0
Height(Inches),0
Weight(Pounds),0


In [5]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)
#splits the data for training and testing - 4 components
#test_size -> the percentage of data to be used for testing

In [6]:
x_train

,Height(Inches)
11143,69.46199
1830,69.15689
5310,70.35624
20066,70.05535
10343,67.23477
...,...
10951,69.51067
9094,69.15991
15058,67.60759
22606,64.90842


In [7]:
from sklearn.linear_model import LinearRegression

model=LinearRegression()
model.fit(x_train,y_train)
# fit -> uses model to train data

LinearRegression()

In [8]:
model.predict(x_test)
#returns predicted data

array([[121.87335809],
       [136.07563464],
       [129.76117783],
       ...,
       [123.47440674],
       [130.69459982],
       [132.25065529]])

In [9]:
y_pred=model.predict(x_test)

In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [11]:
mean_absolute_error(y_test,y_pred)

7.9864062908478415

In [12]:
mean_squared_error(y_test, y_pred)

100.49979051925675

In [13]:
r2_score(y_test, y_pred)

0.24443490024677839

In [14]:
import joblib

joblib.dump(model, 'linear.pkl')

['linear.pkl']

# BIG DATA AND PySpark

In [16]:
!pip install pyspark --quiet

print("Pyspark installation complete")

Pyspark installation complete


In [18]:
from pyspark.sql import SparkSession

from pyspark.sql import functions as F

from pyspark.sql.functions import year, month, to_date, col, round as spark_round

import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spark=SparkSession.builder \
  .appName('Day4_BigData_Sales') \
  .config('Spark.sql.adaptive.enabled', 'true') \
  .getOrCreate()

print(f"Spark version : {spark.version}")
print(f"SparkSession : ACTIVE")
print(f"Application : {spark.sparkContext.appName}")

Spark version : 4.0.2
SparkSession : ACTIVE
Application : Day_BigData_Sales


In [19]:
df_bronze=spark.read \
  .option('header', 'true') \
  .option('interSchema', 'true') \
  .csv('large_sales_data.csv')

print('=== BRONZE LAYER - Raw Data ===')
print(f"Rows : {df_bronze.count()}")
print(f"Columns : {len(df_bronze.columns)}")
print(f"Names : {df_bronze.columns}")
print()
df_bronze.printSchema()

=== BRONZE LAYER - Raw Data ===
Rows : 5000
Columns : 13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [20]:
print("First 5 rows : ")
df_bronze.show(5, truncate=False)

df_bronze.select('quantity', 'unit_price', 'revenue').describe().show()

First 5 rows : 
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi 

In [ ]:
df_bronze.write \
  .mode('overwrite') \
  .parquet('sales_bronze.parquet')

print("Bronze Parquet saved : sales_bronze.parquet")

import os
